### Finding optimal hyperparameters for one of the algorithms (LightGBM in this case)

**For simplicity, second best algorithm (LightGBM) was selected due to simplicity**

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor

import seaborn as sns
import xgboost as xgb
import lightgbm as lgb
import catboost as cb

import tensorflow as tf
import keras
from keras import layers


In [2]:
# load the diamonds dataset, replace this with your own .csv -data
# this version of data should be READY TO GO to training
# which means => only numeric values in the dataset
df = pd.read_csv("processed_regression_diamonds.csv")
df.head()

,carat,cut,color,clarity,table,price,x,y,z
0,0.29,4,2,4,58.0,334,4.20,4.23,2.63
1,0.31,5,1,2,54.0,344,4.35,4.37,2.71
2,0.32,4,6,1,58.0,345,4.38,4.42,2.68
3,0.30,5,2,2,54.0,348,4.31,4.34,2.68
4,0.30,2,1,3,56.0,351,4.23,4.26,2.71


### Same X/y + train/test -splits as in the benchmarking code

In [3]:
# CatBoost requires we save our continuous and categorical variables separately into lists
categorical_variables = ["cut", "color", "clarity"]

# continuous variables also into a list
continuous_variables = ["carat", "table", "price", "depth", "x", "y", "z"]

# the usual X/y -split
X = df.drop("price", axis=1)
y = df['price']

# usual train/test -split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# NOTE! SCALING => some of the algorithms require this
scaler = StandardScaler()

# create separate versions for the scaled data
# because we need both unscaled and scaled versions later
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### In this example, use RandomizedSearchCV to search better hyperparameters for our algorithm

You can try to optimize several algorithms as well! This example only optimizes LightGBM

**NOTE: generating hyperparameter tuning code examples via AI is fine.**

In [4]:
from sklearn.model_selection import RandomizedSearchCV
import numpy as np
from scipy.stats import randint

# Define the parameter distributions with integer ranges
# Notice how we use NumPy and scipy in order to have ranges of values
# for the RandomizedSearch, which it tries to combine in a random manner 
# => based on luck, you might stumble upon a very good combination of parameters
param_dist = {
    'learning_rate': np.linspace(0.08, 0.135, 30), 
    'num_leaves': randint(25, 51),  
    'max_depth': randint(-1, 35), 
}

# Setup RandomizedSearchCV
# search 300 times (n_iter)
# n_jobs => -1 => use all CPU cores
# cv = cross-validation strategy (higher is often better, but takes more time), typical values 3-5
random_search = RandomizedSearchCV(
    estimator=lgb.LGBMRegressor(), 
    param_distributions=param_dist, 
    n_iter=300,  # Number of random combinations to test
    cv=3,
    n_jobs=-1,
    verbose=1,  
    scoring='neg_mean_squared_error'
)

# Fit the randomized search
random_search.fit(X_train, y_train)

# Get the best parameters
best_params_random = random_search.best_params_
print("\nBest parameters from RandomizedSearchCV:", best_params_random)

Fitting 3 folds for each of 300 candidates, totalling 900 fits
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000159 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 930
[LightGBM] [Info] Number of data points in the train set: 33560, number of used features: 8
[LightGBM] [Info] Start training from score 2338.198987

Best parameters from RandomizedSearchCV: {'learning_rate': np.float64(0.12362068965517242), 'max_depth': 25, 'num_leaves': 47}
